# Intelligent NLP-Based Fake News Detection and Classification System
### College Micro-Project Demonstration & Experiments Notebook

**Objective:** Build, evaluate, and optimize an NLP & Machine Learning pipeline to classify news articles as **REAL** or **FAKE**, provide explainable linguistic indicators, and compare multiple models (Naive Bayes, Logistic Regression, Linear SVM, and DistilBERT).

## 1. Imports and Environment Setup

In [ ]:
import os
import sys
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is in sys.path
sys.path.append(os.path.abspath('..'))

from src.preprocessing import preprocess_corpus, preprocess_text
from src.evaluate import compute_metrics, plot_confusion_matrices, plot_model_comparison
from src.explain import explain_prediction
from src.predict import FakeNewsPredictor

%matplotlib inline
sns.set_theme(style="whitegrid")

## 2. Exploratory Data Analysis (EDA)
We load the clean benchmark dataset (`data/news.csv`), inspect class balance, article length distributions, and missing values.

In [ ]:
df = pd.read_csv('../data/news.csv')
print(f"Total Records: {len(df)}")
print("Class Distribution:")
display(df['label'].value_counts())
df.head()

In [ ]:
# Visualizing Class Distribution and Length
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
palette = {'REAL': '#10b981', 'FAKE': '#ef4444'}

sns.countplot(data=df, x='label', palette=palette, ax=axes[0])
axes[0].set_title('Class Balance (REAL vs FAKE)', fontweight='bold')

df['word_count'] = df['combined_text'].apply(lambda x: len(str(x).split()))
sns.histplot(data=df[df['word_count'] <= df['word_count'].quantile(0.95)], x='word_count', hue='label', palette=palette, kde=True, ax=axes[1])
axes[1].set_title('Word Count Distribution (95th Percentile)', fontweight='bold')
plt.tight_layout()
plt.show()

## 3. NLP Preprocessing Pipeline
Our NLP pipeline applies:
1. Case folding (lowercasing)
2. Hyperlink & HTML stripping
3. Punctuation and special character removal
4. Tokenization
5. Stopword filtering (NLTK English stopwords)
6. WordNet Lemmatization (reducing words to canonical base forms)

In [ ]:
sample_raw = "BREAKING NEWS: Shocking miracle cure announced by scientists at http://harvard.edu/test! <p>Officials confirmed results.</p>"
sample_clean = preprocess_text(sample_raw)

print("--- RAW INPUT ---")
print(sample_raw)
print("\n--- AFTER NLP PREPROCESSING ---")
print(sample_clean)

## 4. Train/Test Splitting & TF-IDF Feature Extraction
To prevent data leakage, TF-IDF vectorization is **fitted exclusively on the training set** (80%) and transformed on the test set (20%).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

le = LabelEncoder()
y = le.fit_transform(df['label'])

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    df['combined_text'].values, y, test_size=0.20, random_state=42, stratify=y
)

# Preprocess splits
X_train_clean = preprocess_corpus(X_train_raw)
X_test_clean = preprocess_corpus(X_test_raw)

# TF-IDF Vectorizer with unigrams + bigrams
vectorizer = TfidfVectorizer(max_features=25000, ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True)
X_train_tfidf = vectorizer.fit_transform(X_train_clean)
X_test_tfidf = vectorizer.transform(X_test_clean)

print(f"Training shape: {X_train_tfidf.shape}")
print(f"Testing shape:  {X_test_tfidf.shape}")

## 5. Model Training, Hyperparameter Optimization, and Comparison
We evaluate three core classification algorithms:
1. **Multinomial Naive Bayes** (Fast probabilistic baseline)
2. **Logistic Regression** (Linear log-odds classifier)
3. **Linear SVM** (Calibrated Support Vector Classifier)

In [ ]:
results_df = pd.read_csv('../results/model_comparison.csv')
display(results_df)

In [ ]:
# Display Model Comparison and Confusion Matrix Plots
from IPython.display import Image, display

display(Image(filename='../results/model_comparison.png'))
display(Image(filename='../results/confusion_matrices.png'))

## 6. Explainable AI (XAI) Demonstration
The system extracts the most influential linguistic tokens that drive the model's classification decision.

In [ ]:
predictor = FakeNewsPredictor()

test_article_real = (
    "WASHINGTON (Reuters) - The U.S. Senate on Thursday approved bipartisan legislation "
    "providing disaster relief and infrastructure funds following official committee reports."
)

res_real = predictor.predict(test_article_real)
print(f"Prediction: {res_real['prediction']} ({res_real['confidence']}%) | Model: {res_real['model_used']}")
print(f"Key Indicators: {res_real['important_features']}")
print(f"Explanation: {res_real['explanation']}")

In [ ]:
test_article_fake = (
    "SHOCKING: Secret leaked documents reveal miracle cure hidden by government doctors to protect profits!"
)

res_fake = predictor.predict(test_article_fake)
print(f"Prediction: {res_fake['prediction']} ({res_fake['confidence']}%) | Model: {res_fake['model_used']}")
print(f"Key Indicators: {res_fake['important_features']}")
print(f"Explanation: {res_fake['explanation']}")